# Retinal AV Segmentation: Multiclass (Artery / Vein / Background)

This notebook trains a **MorphoBIFPNUNet** for **3-class segmentation** on fundus images:

| Class | Label colour | Index |
|-------|--------------|-------|
| Background  | Black  | 0 |
| Artery      | Red    | 1 |
| Vein        | Blue   | 2 |

**Dataset layout** (each split folder):
```
Fundus-AVSeg-prep-data/
  images/        *.png          (fundus images)
  annotation/    *.png          (RGB multiclass label maps)
  npy/           *_liot_eg.npy  (LIOT feature arrays)
```

> **Note:** Crossing and Uncertainty pixels in the original annotations are
> remapped to Background (class 0) during loading.


## Imports

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import random
import itertools
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as T

import math
import cv2
from pathlib import Path

import skimage
from skimage.measure import label, regionprops
from skimage import morphology
import torchvision.models as models
from scipy.spatial.distance import pdist, squareform
from codecarbon import EmissionsTracker
import copy, os, time
import matplotlib.gridspec as gridspec

from collections import Counter


print("All imports successful.")

All imports successful.


## Multiclass Constants

In [20]:
NUM_CLASSES   = 3
CLASS_NAMES   = {0: 'Background', 1: 'Artery', 2: 'Vein'}
CLASS_COLORS  = {
    0: np.array([  0,   0,   0], dtype=np.uint8),  # Background : black
    1: np.array([255,   0,   0], dtype=np.uint8),  # Artery     : red
    2: np.array([  0,   0, 255], dtype=np.uint8),  # Vein       : blue
}
VESSEL_CLASSES = [1, 2]   # classes used for mean Dice (exclude background)

print(f"Multiclass segmentation: {NUM_CLASSES} classes")
for k, v in CLASS_NAMES.items():
    print(f"  {k} = {v}")


Multiclass segmentation: 3 classes
  0 = Background
  1 = Artery
  2 = Vein


## Dataset Class

Loads PNG fundus images, LIOT `.npy` feature arrays, and converts RGB annotation maps to integer class masks.

In [ ]:
class ImageDataset(data.Dataset):
    """
    Multiclass retinal AV segmentation dataset using RGB images only.

    Every image (and its annotation mask) is resized to (new_size x new_size)
    so that batching works regardless of the original resolution.

    Expected layout per split root:
        <root>/
            images/       *.png           (fundus images, RGB)
            annotation/   *.png           (RGB multiclass label maps)

    Label colour → class index:
        black  (0,0,0)       → 0  background
        red    (255,0,0)     → 1  artery
        blue   (0,0,255)     → 2  vein
        green  (0,255,0)     → 3  crossing   (remapped to 0)
        white  (255,255,255) → 4  uncertainty (remapped to 0)
    """

    def __init__(self, root, new_size=592, supervised=True):
        self.root       = Path(root)
        self.supervised = supervised
        self.new_size   = new_size

        img_dir = self.root / 'images'
        ann_dir = self.root / 'annotation'

        self.image_paths = sorted(img_dir.glob('*.png'))
        self.label_paths = []

        if self.supervised:
            for img_path in self.image_paths:
                stem = img_path.stem
                self.label_paths.append(ann_dir / f'{stem}.png')

        print(f"Dataset root        : {root}")
        print(f"  Images found      : {len(self.image_paths)}")
        if self.supervised:
            print(f"  Annotations found : {len(self.label_paths)}")
        if self.image_paths:
            print(f"  First 5 images    : {[p.name for p in self.image_paths[:5]]}")

    # ── RGB annotation → integer class mask ───────────────────────────────────

    @staticmethod
    def rgb_to_class_mask(rgb_array):
        """
        Convert an RGB annotation array (H, W, 3) to an integer class mask (H, W).

        Crossing (green) and Uncertainty (white) pixels are remapped to
        Background (0) — only 3 classes are used.
        """
        r, g, b = rgb_array[:, :, 0], rgb_array[:, :, 1], rgb_array[:, :, 2]
        mask = np.zeros(rgb_array.shape[:2], dtype=np.int64)
        mask[(r > 180) & (g < 80)  & (b < 80)]  = 1   # Artery
        mask[(r < 80)  & (g < 80)  & (b > 180)] = 2   # Vein
        return mask

    # ── Dataset protocol ──────────────────────────────────────────────────────

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        s = self.new_size

        # ── Fundus image: resize to (s, s) ───────────────────────────────────
        image = Image.open(self.image_paths[index]).convert('RGB')
        image = image.resize((s, s), Image.BILINEAR)
        image = T.ToTensor()(image)                         # (3, s, s)

        # ── Annotation → integer class mask, resize with NEAREST ─────────────
        if self.supervised:
            ann_rgb = np.array(
                Image.open(self.label_paths[index])
                     .convert('RGB')
                     .resize((s, s), Image.NEAREST)
            )                                               # (s, s, 3)
            mask  = self.rgb_to_class_mask(ann_rgb)         # (s, s) int64
            label = torch.from_numpy(mask)                  # LongTensor (s, s)
            return image, label

        return image


## Load Data

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
root = r"/home/mefeki/Bureau/work/Fundus-AVSeg-prep-data"

test_dataset              = ImageDataset(root=os.path.join(root, 'test'))
train_dataset_supervised  = ImageDataset(root=os.path.join(root, 'train'))

train_dataloader_supervised = data.DataLoader(
    train_dataset_supervised, shuffle=True, batch_size=2)
test_dataloader = data.DataLoader(test_dataset, shuffle=False, batch_size=1)

dataloaders   = {'train supervised': train_dataloader_supervised,
                 'test':             test_dataloader}
dataset_sizes = {'train supervised': len(train_dataset_supervised),
                 'test':             len(test_dataset)}
batch_sizes   = {'train supervised': 2, 'test': 1}

print("Dataset sizes:", dataset_sizes)

# ── Quick sanity-check visualisation ──────────────────────────────────────────
it = iter(dataloaders['test'])
inputs, labels = next(it)

print("Image shape     :", inputs.shape)
print("Label shape     :", labels.shape, "  dtype:", labels.dtype)
print("Unique classes  :", labels.unique().tolist())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# RGB image
out_inputs = torchvision.utils.make_grid(inputs)
axes[0].imshow(out_inputs.permute(1, 2, 0).clamp(0, 1))
axes[0].set_title('Fundus Image')
axes[0].axis('off')

# Colour-coded ground-truth label
lbl_np  = labels[0].numpy()
lbl_rgb = np.zeros((*lbl_np.shape, 3), dtype=np.uint8)
for cls_idx, color in CLASS_COLORS.items():
    lbl_rgb[lbl_np == cls_idx] = color
axes[1].imshow(lbl_rgb)
axes[1].set_title('Annotation (colour-coded)')
axes[1].axis('off')

from matplotlib.patches import Patch
legend_handles = [
    Patch(color=np.array(c)/255, label=n)
    for c, n in zip(CLASS_COLORS.values(), CLASS_NAMES.values())
]
axes[1].legend(handles=legend_handles, loc='lower right', fontsize=7)

fig.tight_layout()
plt.show()


## Colour-Map Utility

In [23]:
def class_to_rgb(mask_np):
    """
    Convert an integer class mask (H, W) to a colour-coded RGB image (H, W, 3).

    Parameters
    ----------
    mask_np : np.ndarray, shape (H, W), dtype int
        Pixel-wise class indices (0=Background, 1=Artery, 2=Vein).

    Returns
    -------
    rgb : np.ndarray, shape (H, W, 3), dtype uint8
    """
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for cls_idx, color in CLASS_COLORS.items():
        rgb[mask_np == cls_idx] = color
    return rgb

## Network Architecture

The architecture is unchanged except for the **output head**, which now produces `NUM_CLASSES = 3` logit maps instead of 1.


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class StandardUNet(nn.Module):
    def __init__(self, in_channels=4, n_classes=3):
        super(StandardUNet, self).__init__()
        self.in_channels = in_channels
        self.n_classes = n_classes

        # Encoder
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        
        # Decoder
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)
        
        # Output
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5)
        diffY = x4.size()[2] - x.size()[2]
        diffX = x4.size()[3] - x.size()[3]
        x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x4, x], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        diffY = x3.size()[2] - x.size()[2]
        diffX = x3.size()[3] - x.size()[3]
        x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x3, x], dim=1)
        x = self.conv2(x)
        
        x = self.up3(x)
        diffY = x2.size()[2] - x.size()[2]
        diffX = x2.size()[3] - x.size()[3]
        x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x], dim=1)
        x = self.conv3(x)
        
        x = self.up4(x)
        diffY = x1.size()[2] - x.size()[2]
        diffX = x1.size()[3] - x.size()[3]
        x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x1, x], dim=1)
        x = self.conv4(x)
        
        logits = self.outc(x)
        return logits

## CLAHE Preprocessing

In [25]:
def clahe(inputs, clipLimit=4, tileGridSize=(8, 8)):
    """Apply CLAHE to the green channel; returns (B, 1, H, W) float tensor."""
    inputs      = inputs.to('cpu')
    clahe_obj   = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    green_ch    = np.uint8(255 * inputs[:, 1, :, :])
    res         = np.array([clahe_obj.apply(g) for g in green_ch], dtype=float) / 255
    return torch.tensor(res, dtype=torch.float).unsqueeze(1).to(device)

## Multiclass Boundary Loss (MulticlassBZLoss)

Adapted from the binary `BZ_Loss` to handle 3-class output:

* **CrossEntropyLoss** replaces BCE (accepts raw logits + integer targets).
* **BZ boundary term** is applied to the foreground probability `P_fg = 1 − P(background)`, which encodes the combined vessel signal.
* **CL-Dice** is computed per vessel class (artery, vein) and averaged.


In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# Weighted Cross-Entropy Loss
# ══════════════════════════════════════════════════════════════════════════════
class WeightedCELoss(nn.Module):
    """
    Standard Weighted Cross-Entropy Loss for multiclass segmentation.
    Weights are used to balance classes (e.g., if Background is overrepresented).
    """
    def __init__(self, class_weights=None):
        super(WeightedCELoss, self).__init__()
        # Ensure class_weights is a tensor moved to the correct device
        self.criterion = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, outputs, labels):
        """
        outputs: (B, 3, H, W)
        labels:  (B, H, W) containing class indices (0, 1, 2)
        """
        # CrossEntropyLoss expects labels as (B, H, W) with class indices
        return self.criterion(outputs, labels.long())

## Evaluation Metrics

All functions accept `(B, H, W)` LongTensor predictions and labels.

In [27]:
def accuracy(preds, labels, eps=1e-6):
    """Overall pixel accuracy."""
    preds  = original_size(preds)
    labels = original_size(labels)
    return float(torch.sum(preds == labels)) / (float(labels.numel()) + eps)


def dice_coef_per_class(preds, labels, num_classes=NUM_CLASSES, eps=1e-6):
    """
    Compute Dice coefficient for each class.

    Returns
    -------
    per_class : dict {class_idx: dice_value}
    mean_vessel_dice : float
        Mean Dice over VESSEL_CLASSES = [1 (artery), 2 (vein)].
    """
    preds  = original_size(preds)
    labels = original_size(labels)
    per_class = {}
    for c in range(num_classes):
        pred_c  = (preds  == c).float()
        label_c = (labels == c).float()
        tp = float(torch.sum(pred_c  * label_c))
        fp = float(torch.sum(pred_c  * (1 - label_c)))
        fn = float(torch.sum((1 - pred_c) * label_c))
        per_class[c] = 2 * tp / (2 * tp + fp + fn + eps)
    mean_vessel_dice = float(np.mean([per_class[c] for c in VESSEL_CLASSES]))
    return per_class, mean_vessel_dice


def sensitivity(preds, labels, cls=1, eps=1e-6):
    """Sensitivity (recall) for a given class."""
    preds  = original_size(preds)
    labels = original_size(labels)
    TP = float(torch.sum((preds == cls) & (labels == cls)))
    FN = float(torch.sum((preds != cls) & (labels == cls)))
    return TP / (TP + FN + eps)


def specificity(preds, labels, cls=1, eps=1e-6):
    """Specificity for a given class."""
    preds  = original_size(preds)
    labels = original_size(labels)
    TN = float(torch.sum((preds != cls) & (labels != cls)))
    FP = float(torch.sum((preds == cls) & (labels != cls)))
    return TN / (TN + FP + eps)


def precision(preds, labels, cls=1, eps=1e-6):
    """Precision for a given class."""
    preds  = original_size(preds)
    labels = original_size(labels)
    TP = float(torch.sum((preds == cls) & (labels == cls)))
    FP = float(torch.sum((preds == cls) & (labels != cls)))
    return TP / (TP + FP + eps)

## Post-processing: Remove Circular FOV Boundary

In [28]:
def remove_circle(inputs, preds, kernel_size, threshold):
    """
    Zero-out (set to background class 0) predictions that fall outside the
    circular field-of-view inferred from the fundus image brightness.

    Parameters
    ----------
    inputs : Tensor (B, C, H, W)   original RGB fundus image (float).
    preds  : Tensor (B, H, W)      integer class predictions (LongTensor).
    kernel_size : int              morphological line-erosion kernel size.
    threshold   : float            brightness threshold for FOV detection.

    Returns
    -------
    masked_preds : Tensor (B, H, W)  LongTensor, 0 outside FOV.
    """
    # Handle RGBA: composite on white background
    if inputs.shape[1] == 4:
        rgb   = inputs[:, :3, :, :]
        alpha = inputs[:, 3:, :, :]
        inputs_rgb = rgb * alpha + (1 - alpha)
    else:
        inputs_rgb = inputs

    RGB_to_GRAY = T.Grayscale()
    mask = (RGB_to_GRAY(inputs_rgb) > threshold).float()   # (B, 1, H, W)

    p1 = -F.max_pool2d(-mask, kernel_size=(kernel_size, 1), stride=(1, 1),
                       padding=(kernel_size // 2, 0))
    p2 = -F.max_pool2d(-mask, kernel_size=(1, kernel_size), stride=(1, 1),
                       padding=(0, kernel_size // 2))

    circle_mask = (torch.min(p1, p2) > 0).squeeze(1).long()  # (B, H, W)

    return preds * circle_mask   # 0 outside FOV → background

## Training & Test Function

In [ ]:
def train_model(model, criterion, optimizer, dataloader, num_epochs):
    model.train()
    epoch_losses = []

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        batch_losses = []
        for inputs, labels in dataloader:
            inputs = inputs.to(device)          # (B, 3, H, W) RGB
            labels = labels.to(device)

            # Augment with CLAHE green channel → (B, 4, H, W)
            clahe_ch = clahe(inputs)            # (B, 1, H, W)
            model_input = torch.cat([inputs, clahe_ch], dim=1)

            optimizer.zero_grad()
            outputs = model(model_input)

            loss = criterion(outputs, labels.long())
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        mean_loss = sum(batch_losses) / len(batch_losses)
        epoch_losses.append(mean_loss)
        print(f'Loss: {mean_loss:.4f}')

    print('Training complete.')
    return model, epoch_losses


def original_size(tensor):
    return tensor


def dice_coef_per_class(preds, labels, num_classes=NUM_CLASSES, eps=1e-6):
    preds  = original_size(preds)
    labels = original_size(labels)
    per_class = {}
    for c in range(num_classes):
        pred_c  = (preds  == c).float()
        label_c = (labels == c).float()
        tp = float(torch.sum(pred_c  * label_c))
        fp = float(torch.sum(pred_c  * (1 - label_c)))
        fn = float(torch.sum((1 - pred_c) * label_c))
        per_class[c] = 2 * tp / (2 * tp + fp + fn + eps)
    mean_vessel_dice = float(np.mean([per_class[c] for c in VESSEL_CLASSES]))
    return per_class, mean_vessel_dice


def test_model(model, dataloader):
    model.eval()
    total_dice = 0.0
    n = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            clahe_ch    = clahe(inputs)
            model_input = torch.cat([inputs, clahe_ch], dim=1)

            logits = model(model_input)
            probs  = torch.softmax(logits, dim=1)
            preds  = torch.argmax(probs, dim=1)

            _, vessel_dice = dice_coef_per_class(preds, labels)
            total_dice += vessel_dice
            n += 1

    final_dice = total_dice / max(n, 1)
    print(f'\nMean vessel Dice on test set: {final_dice:.4f}')
    print('For reference, models outperformed another model: Unet dice 0.739')
    return final_dice


## Fixed 8-Image Subset: indices [1, 3, 5, 6, 7, 12, 15, 16]

The training subset is fixed to the 8 images at indices `[1, 3, 5, 6, 7, 12, 15, 16]`. The model will be trained directly on this subset for **150 epochs**.

In [30]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Load full training dataset ────────────────────────────────────────────────
train_root         = os.path.join(root, 'train')
train_dataset_full = ImageDataset(root=train_root, supervised=True)

print(f"Total training images : {len(train_dataset_full)}")
print('Using full training set. ✓')


## class weights

## Training on the Fixed Subset (150 epochs)

A fresh **MorphoBIFPNUNet** (multiclass, 3 outputs) is trained from scratch on the selected images for **150 epochs**.


In [ ]:
# ── Training hyper-parameters ─────────────────────────────────────────────────
EPOCHS     = 150
BATCH_SIZE = 2
SEED       = 42

full_dataloader = data.DataLoader(
    train_dataset_full, shuffle=True, batch_size=BATCH_SIZE)

print(f'\n=== Training on full training set ({len(train_dataset_full)} images) ===')

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

# ── Class weights (computed over full training set) ────────────────────────────
pixel_counts = Counter()
for _, lbl in train_dataset_full:
    for c in range(NUM_CLASSES):
        pixel_counts[c] += int((lbl == c).sum())

total    = sum(pixel_counts.values())
freqs    = torch.tensor([pixel_counts[c] / total for c in range(NUM_CLASSES)])
inv_freq = 1.0 / (freqs + 1e-6)
class_weights = inv_freq / inv_freq[1:].mean()   # normalise by vessel mean

print("Class pixel %:", {CLASS_NAMES[c]: f"{100*freqs[c].item():.2f}%" for c in range(NUM_CLASSES)})
print("Class weights:", class_weights)
class_weights = class_weights.to(device)

# RGB (3) + CLAHE green (1) = 4 input channels
model     = StandardUNet(in_channels=4, n_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3,
                       betas=(0.9, 0.999), eps=1e-07)

model, training_losses = train_model(model, criterion, optimizer, full_dataloader, num_epochs=EPOCHS)

# Evaluate
best_dice = test_model(model, test_dataloader)

# Save
os.makedirs('models', exist_ok=True)
save_path = 'models/best_model_morpho_unet.pth'
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

# Plot diagnostics
plt.figure(figsize=(8, 6))
plt.plot(range(1, EPOCHS+1), training_losses, label='Train Loss')
plt.xlabel('Epoch')
plt.ylabel('Weighted Cross Entropy Loss')
plt.title(f'Morpho U-Net Training (full set, RGB+CLAHE) | Final Dice: {best_dice:.4f}')
plt.legend()
plt.grid(True)
plt.savefig('training_diagnostics_morpho_unet.png', dpi=150)
plt.show()


## (Optional) Visualise Best Model Predictions

In [ ]:
model.eval()
with torch.no_grad():
    print(f"{'Class':<15} {'Dice':>8}")
    print('-' * 25)

    all_class_dice = {c: [] for c in range(NUM_CLASSES)}

    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)

        clahe_ch    = clahe(inputs)
        model_input = torch.cat([inputs, clahe_ch], dim=1)

        logits = model(model_input)
        preds  = torch.argmax(torch.softmax(logits, dim=1), dim=1)

        per_class, _ = dice_coef_per_class(preds, labels)
        for c, d in per_class.items():
            all_class_dice[c].append(d)

    for c in range(NUM_CLASSES):
        mean_d = np.mean(all_class_dice[c])
        print(f"  {CLASS_NAMES[c]:<13} {mean_d:>8.4f}")

    mean_vessel = np.mean([np.mean(all_class_dice[c]) for c in VESSEL_CLASSES])
    print(f"\n  Mean vessel Dice (artery + vein): {mean_vessel:.4f}")


def display_test_batch(model, dataloader):
    model.eval()
    inputs, labels = next(iter(dataloader))
    with torch.no_grad():
        inputs_dev  = inputs.to(device)
        clahe_ch    = clahe(inputs_dev)
        model_input = torch.cat([inputs_dev, clahe_ch], dim=1)
        logits = model(model_input)
        preds  = torch.argmax(torch.softmax(logits, dim=1), dim=1).cpu()

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(inputs[0].permute(1, 2, 0))
    ax[0].set_title("Input Image")
    ax[1].imshow(labels[0].cpu(), cmap='jet')
    ax[1].set_title("Ground Truth")
    ax[2].imshow(preds[0], cmap='jet')
    ax[2].set_title("Prediction")
    plt.show()

display_test_batch(model, dataloaders['test'])
